# Clasificación de tumores: maligno vs. benigno

**Problema:** dado un tumor descrito por 30 medidas, decidir si es maligno o benigno.
**Lo que importa:** un maligno clasificado como benigno (falso negativo) es el error más
grave; un benigno clasificado como maligno solo lleva a más estudios. Con esto voy a elegir
la métrica y el modelo.

### 1. Análisis exploratorio de datos (EDA)

In [ ]:
from sklearn.datasets import load_breast_cancer

In [ ]:
import pandas as pd # sec. 1
from sklearn.model_selection import train_test_split # sec. 2
from sklearn.preprocessing import StandardScaler # sec. 3
from sklearn.linear_model import LogisticRegression # sec. 4
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

In [3]:
data = load_breast_cancer(as_frame=True)
X, y  = data.data, data.target

In [4]:
X

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


In [5]:
y

0      0
1      0
2      0
3      0
4      0
      ..
564    0
565    0
566    0
567    0
568    1
Name: target, Length: 569, dtype: int64

In [ ]:
# target: 0 = maligno, 1 = benigno
print(X.shape)
print(y.value_counts(normalize=True).round(2))

(569, 30)
target
1    0.63
0    0.37
Name: proportion, dtype: float64


In [16]:
print(X.isna().sum().sum())

X.describe().T[["mean", "std", "min", "max"]].sort_values("mean")

0


,mean,std,min,max
fractal dimension error,0.003795,0.002646,0.000895,0.02984
smoothness error,0.007041,0.003003,0.001713,0.03113
concave points error,0.011796,0.006170,0.000000,0.05279
symmetry error,0.020542,0.008266,0.007882,0.07895
compactness error,0.025478,0.017908,0.002252,0.13540
concavity error,0.031894,0.030186,0.000000,0.39600
mean concave points,0.048919,0.038803,0.000000,0.20120
mean fractal dimension,0.062798,0.007060,0.049960,0.09744
worst fractal dimension,0.083946,0.018061,0.055040,0.20750
mean concavity,0.088799,0.079720,0.000000,0.42680


#### Lo que encontré
- 569 tumores, 30 medidas, sin valores faltantes.
- 212 malignos (37 %) y 357 benignos (63 %). Un modelo que siempre diga «benigno»
  tendría 63 % de accuracy sin detectar ningún maligno: por eso el accuracy no basta.
- Las escalas son muy distintas (de ~0.004 a ~880). Los modelos sensibles a la escala
  necesitan estandarizar; los árboles no.

### 2. Partición de datos: entrenamiento, validación y prueba

In [ ]:
# Datos (569) ──┬── train 60%  → aprenden los modelos
#               └── temp  40%  ──┬── val   50% → comparo y elijo
#                                └── test  50%  → solo al final



#Conjunto	    %	Función

# Train	      60%	Entrenar el modelo (ajustar parámetros)
# Validation  20%	Ajustar hiperparámetros / elegir modelo / early stopping
# Test	      20%	Evaluación final honesta (nunca se toca durante el desarrollo)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, 
	test_size=0.4, 
	stratify=y, 
	random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
	test_size=0.5, 
	stratify=y_temp, 
	random_state=42
)

print(len(X_train), len(X_val), len(X_test))

341 114 114


### 3. Preprocesamiento

In [27]:
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print(X_train_s.mean().round(2), X_train_s.std().round(2))

-0.0 1.0


### 4. Entrenamiento de modelos

In [28]:
models = {
	"Logistic Regression": (LogisticRegression(max_iter=1_000), X_train_s, X_val_s),
	"KNN": (KNeighborsClassifier(n_neighbors=5), X_train_s, X_val_s),
	"Random Forest": (RandomForestClassifier(random_state=42), X_train, X_val)
}

In [29]:
for name, (model, Xtr, Xv) in models.items():
	model.fit(Xtr, y_train)
	print(f"Modelo {name} Entrenado ")

Modelo Logistic Regression Entrenado 
Modelo KNN Entrenado 
Modelo Random Forest Entrenado 
